# Backpropagation and Neural Network Fitting

This notebook:
- Performs one manual update for a single sigmoid neuron (MSE).
- Performs one manual update for a 2–2–1 sigmoid network (MSE).
- Trains a neural network to fit a binary initial image using Keras.

---
# Problem 1 — Single Neuron (Sigmoid + MSE)

Model:

$z = \theta_0 + \theta_1 x_1 + \theta_2 x_2$

$\hat{y} = \sigma(z) = \frac{1}{1+e^{-z}}$

$L = \frac{1}{2}(\hat{y} - y)^2$

Goal: perform one weight update.

In [ ]:
import numpy as np

# Given values
x1, x2 = 1, -1
theta = np.array([0.1, 0.4, -0.3], dtype=float)
alpha = 0.5
y_true = 1

def sigmoid(z):  # activation
    return 1 / (1 + np.exp(-z))

## Forward Pass

In [ ]:
z = theta[0] + theta[1]*x1 + theta[2]*x2
y_hat = sigmoid(z)  # activation
loss = 0.5 * (y_hat - y_true)**2  # compute loss

print('z =', z)
print('y_hat =', y_hat)
print('loss =', loss)  # compute loss

## Backward Pass

$\frac{\partial L}{\partial z} = (\hat{y}-y)\hat{y}(1-\hat{y})$

In [ ]:
dL_dz = (y_hat - y_true) * y_hat * (1 - y_hat)
grad = np.array([dL_dz * 1,
                 dL_dz * x1,
                 dL_dz * x2])

theta_new = theta - alpha * grad

print('Updated weights:', theta_new)

---
# Problem 2 — 2–2–1 Network (Sigmoid + MSE)

Each neuron:
$z = w \cdot x$,  $a = \sigma(z)$

Loss: $L = \frac{1}{2}(\hat{y} - y)^2$

Perform one full update for all 9 weights.

In [ ]:
# Inputs (bias, x1, x2)
x = np.array([1, 1, 0], dtype=float)
y_true = 1

# Hidden weights
W_hidden = np.array([
    [1, 1, 1],
    [-1, -1, -1]
], dtype=float)

# Output weights
v = np.array([1, -1, 1], dtype=float)

alpha = 0.5

## Forward Pass

In [ ]:
z_hidden = W_hidden @ x
a_hidden = sigmoid(z_hidden)  # activation

x_out = np.array([1, a_hidden[0], a_hidden[1]])
z_out = v @ x_out
y_hat = sigmoid(z_out)  # activation
loss = 0.5 * (y_hat - y_true)**2  # compute loss

print('Hidden activations:', a_hidden)
print('Output:', y_hat)
print('Loss:', loss)

## Backpropagation

$\delta_3 = (\hat{y}-y)\hat{y}(1-\hat{y})$

$\delta_i = \delta_3 v_i a_i(1-a_i)$

In [ ]:
delta_out = (y_hat - y_true) * y_hat * (1 - y_hat)
delta_hidden = delta_out * v[1:] * a_hidden * (1 - a_hidden)

# Gradients
grad_v = delta_out * x_out
grad_W_hidden = np.zeros_like(W_hidden)
grad_W_hidden[0] = delta_hidden[0] * x
grad_W_hidden[1] = delta_hidden[1] * x

# Update
v_new = v - alpha * grad_v
W_hidden_new = W_hidden - alpha * grad_W_hidden

print('Updated output weights:', v_new)
print('Updated hidden weights:\n', W_hidden_new)

---
# Problem 3 — Fit a Binary Initial

Learn a function:
$f(x_1, x_2) \approx$ pixel intensity (0 or 1)

Inputs: normalized row and column coordinates.
Output: probability pixel is white.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Create simple 'L'
H, W = 32, 32
img = np.zeros((H, W))
img[:, 2:4] = 1
img[-4:-2, :] = 1

# Dataset
X = []
y = []
for r in range(H):
    for c in range(W):
        X.append([r/(H-1), c/(W-1)])
        y.append(img[r, c])

X = np.array(X)
y = np.array(y)

## Model

A multilayer perceptron (fully connected network):
2 inputs → hidden layers → 1 sigmoid output.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy'  # compute loss
)

## Train

In [ ]:
model.fit(X, y, epochs=60, batch_size=5, verbose=0)

## Evaluate

In [ ]:
pred = model.predict(X, verbose=0)
img_pred = pred.reshape(H, W)
img_bin = (img_pred >= 0.5).astype(float)

plt.figure()
plt.imshow(img, cmap='gray')
plt.title('Target')
plt.axis('off')

plt.figure()
plt.imshow(img_bin, cmap='gray')
plt.title('Model Output (Thresholded)')
plt.axis('off')